# Build Your Own RAG System (Google Colab)

**Goal:** By the end, you will have a working Retrieval-Augmented Generation (RAG) system that answers questions from documents.

## How to use this notebook
Run cells **top to bottom**, one section at a time. Read the markdown first, then run the code.

**No local setup required** — Colab provides Python and compute. You only need a free [Gemini API key](https://aistudio.google.com/apikey).

## What you will learn
1. What embeddings are
2. How to load and chunk documents
3. How to store chunks in a vector database
4. How to retrieve relevant context and ask an LLM to answer


## Section 1: Setup

**Run the two cells below:**
1. **Bootstrap** — downloads the workshop repo, installs packages, and sets your API key.
2. **Verify** — confirms everything is ready.

### API key (recommended: Colab Secrets)
1. Click the **key icon** in the left sidebar (Secrets).
2. Add a secret named `GEMINI_API_KEY` with your key from [Google AI Studio](https://aistudio.google.com/apikey).
3. Toggle **Notebook access** on for that secret.

If you skip Secrets, the bootstrap cell will prompt you to paste the key instead.

**First run:** bootstrap may take a few minutes while packages and the embedding model download.

In [ ]:
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

REPO_URL = "https://github.com/Olamilekan002/rag-workshop.git"
REPO_DIR = Path("/content/rag-workshop")

ROOT = Path.cwd()
if (ROOT / "rag_pipeline.py").exists():
    pass
elif ROOT.name == "notebooks" and (ROOT.parent / "rag_pipeline.py").exists():
    ROOT = ROOT.parent
elif REPO_DIR.exists() and (REPO_DIR / "rag_pipeline.py").exists():
    ROOT = REPO_DIR
else:
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    ROOT = REPO_DIR

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

requirements = ROOT / "requirements.txt"
if not requirements.exists():
    raise FileNotFoundError(f"Could not find requirements.txt at {requirements}")

colab_only = {"jupyter", "ipykernel", "streamlit"}
packages = [
    line.strip()
    for line in requirements.read_text().splitlines()
    if line.strip() and not line.strip().startswith("#")
    and line.split(">=")[0].split("==")[0].strip().lower() not in colab_only
]

print("Installing dependencies (first run may take a few minutes)...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
)

api_key = ""
try:
    from google.colab import userdata

    api_key = userdata.get("GEMINI_API_KEY").strip()
    print("Loaded GEMINI_API_KEY from Colab Secrets.")
except Exception:
    print("Colab Secret not found. Paste your key below.")
    print("Get a free key: https://aistudio.google.com/apikey")
    api_key = getpass("GEMINI_API_KEY (input hidden): ").strip()

if not api_key or api_key == "your_key_here":
    raise ValueError("GEMINI_API_KEY is required. Add it in Colab Secrets or paste when prompted.")

os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_API_KEY"] = api_key

env_path = ROOT / ".env"
env_path.write_text(f"LLM_PROVIDER=gemini\nGEMINI_API_KEY={api_key}\n")

print("Bootstrap complete.")
print("Repo root:", ROOT)

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from workshop_env import suppress_workshop_warnings; suppress_workshop_warnings()

from dotenv import load_dotenv

load_dotenv(ROOT / ".env", override=True)

provider = os.getenv("LLM_PROVIDER", "gemini").strip().lower()
api_key = os.getenv("GEMINI_API_KEY")
if provider != "gemini":
    raise ValueError("Set LLM_PROVIDER=gemini for this workshop.")
if not api_key or api_key == "your_key_here":
    raise ValueError("Run the bootstrap cell above to set GEMINI_API_KEY.")

print("Setup OK")
print("LLM provider:", provider)
print("Repo root:", ROOT)

## Section 2: What Are Embeddings?

An **embedding** turns text into a list of numbers that capture meaning.

- Similar meanings -> similar numbers
- This is how search works in modern AI

We will compare three sentences and see which pair is most similar.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import numpy as np

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "The student passed the examination",
    "The learner succeeded in the test",
    "The banana is yellow",
]

vectors = embeddings.embed_documents(sentences)


def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print(
    "Similarity (exam sentence vs test sentence):",
    round(cosine_similarity(vectors[0], vectors[1]), 3),
)
print(
    "Similarity (exam sentence vs banana sentence):",
    round(cosine_similarity(vectors[0], vectors[2]), 3),
)

## Section 3: Load Documents

RAG starts with your source documents.

We will load the sample files in `sample_docs/`:

| File | Topic |
|------|-------|
| `course_syllabus.txt` | Sample university syllabus |
| `nigerian_constitution_excerpt.txt` | Government and welfare (constitution excerpt) |
| `research_paper_excerpt.txt` | Sample research abstract |
| `ifs415_course_material.pdf` | IFS 415 course material (PDF) |

The pipeline loads `.txt` and `.pdf` files. For this workshop we use **structured `.txt` files** because they chunk and retrieve more reliably than scanned or messy PDFs.

**Try this after the demo:** upload your own `.txt` file to `sample_docs/` in the file browser (left sidebar), rebuild the vector store, and ask questions about it.

### Hint (Section 3)

Work through these steps before you code:

1. Open `rag_pipeline.py` in the repo root and find the function that **loads files from a folder**.
2. That function returns a **list of Document objects** (one per file or page).
3. Pass it the path to our sample folder: combine `ROOT` with `"sample_docs"`.
4. Assign the result to a variable named `documents`.

**Check yourself:** after running, `len(documents)` should be at least **3**, and `documents[0].page_content` should contain readable text.

In [ ]:
# YOUR CODE HERE
# Hint: use rag_pipeline.load_documents(ROOT / "sample_docs")

from rag_pipeline import load_documents

documents = None  # replace this

print("Loaded documents:", len(documents))
print("First document preview:")
print(documents[0].page_content[:300])

## Section 4: Chunk the Documents

LLMs cannot read a whole book at once. We split documents into smaller **chunks**.

Good defaults for this workshop:
- `chunk_size=1000`
- `chunk_overlap=100`

### Hint (Section 4)

You already have `documents` from the previous step.

1. Find the **splitting** function in `rag_pipeline.py`.
2. It takes your document list and returns smaller **chunks**.
3. Default workshop settings are already built in (`chunk_size=1000`, `chunk_overlap=100`) — you only need to pass `documents`.
4. Assign the result to `chunks`.

**Check yourself:** you should have **more chunks than documents**. If counts are equal, re-read the chunking section above.

In [ ]:
# YOUR CODE HERE
# Hint: use rag_pipeline.split_documents(documents)

from rag_pipeline import split_documents

chunks = None  # replace this

print("Number of chunks:", len(chunks))
print("Sample chunk:")
print(chunks[0].page_content[:300])

## Section 5: Build the Vector Store (ChromaDB)

Now we:
1. Convert each chunk to an embedding
2. Save embeddings in a local database called **ChromaDB**

In Colab this is stored in your runtime session (temporary unless you download it).

### Hint (Section 5)

This step connects chunking to storage:

1. Find the function that **builds a vector store** in `rag_pipeline.py`.
2. It needs two things: your `chunks` and a **folder path** where ChromaDB will save data.
3. Use `ROOT / "chroma_db"` as the save location.
4. Assign the returned vector store object to `vector_store`.

**Check yourself:** after running, you should see a `chroma_db/` folder under the repo root.

In [ ]:
# YOUR CODE HERE
# Hint: use rag_pipeline.build_vector_store(chunks, ROOT / "chroma_db")

from rag_pipeline import build_vector_store

vector_store = None  # replace this

print("Vector store created at:", ROOT / "chroma_db")

## Section 6: Retrieve Relevant Chunks

When a user asks a question, RAG searches the vector store for the most relevant chunks.

Run a test question and inspect what comes back.

### Hint (Section 6)

No new imports needed — use the `vector_store` object from Section 5.

1. Look for a **similarity search** method on the vector store object.
2. Pass the `question` string already defined in the cell.
3. Set `k=6` to retrieve the top 6 most relevant chunks.
4. Assign the result to `retrieved_docs`.

**Check yourself:** each item in `retrieved_docs` should have `.page_content` and `.metadata["source"]`.

In [ ]:
# YOUR CODE HERE
question = "What is the primary purpose of government in Nigeria?"

retrieved_docs = None  # replace with vector_store.similarity_search(question, k=6)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Chunk {i} | Source: {doc.metadata.get('source')} ---")
    print(doc.page_content[:250])
    print()

## Section 7: Build the RAG Answer

Now we combine:
- Retrieved context
- User question
- LLM generation

This is the core RAG pattern used in ChatGPT plugins, Perplexity, and many enterprise tools.

### Hint (Section 7)

This is the full RAG step — retrieval plus generation:

1. Import `answer_question` from `rag_pipeline`.
2. Call it with `vector_store` and the same `question` from Section 6.
3. It returns a **dictionary** with keys like `"answer"` and `"sources"`.
4. Assign that dictionary to `result`.

**Check yourself:** `result["answer"]` should mention government or welfare if retrieval worked. `result["sources"]` should list filenames.

In [ ]:
# YOUR CODE HERE
from rag_pipeline import answer_question

result = None  # replace this

print("Question:", question)
print("Answer:", result["answer"])
print("Sources:", result["sources"])

## Section 8: Compare With and Without RAG

Ask the LLM directly without context. Then compare with your RAG answer.

This shows why retrieval matters.

In [ ]:
from rag_pipeline import get_llm

llm = get_llm()
plain_answer = llm.invoke(question)
print("Without RAG:\n", plain_answer.content)
print("\nWith RAG:\n", result["answer"])

## Section 9: Ask Your Own Questions

On a local machine, the repo includes a Streamlit chat app (`streamlit run app.py`).

In Colab, use the helper below to ask more questions against your vector store.

**Workshop stretch task:** Upload your own `.txt` document to `sample_docs/`, re-run Sections 5–9, and test 3 questions from your field.

In [ ]:
from rag_pipeline import answer_question


def ask_rag(user_question: str) -> None:
    response = answer_question(vector_store, user_question)
    print("Question:", user_question)
    print("Answer:", response["answer"])
    print("Sources:", response["sources"])
    print()


ask_rag("What topics are covered in the data science course?")
ask_rag("What does the research paper say about transformer models?")

# Change this and re-run:
my_question = "Your question here"
ask_rag(my_question)